# Explainable Feature Creations
### We are running in Google Colab
### A few key explainable features in the HAI analysis of student process project      
 The project website:  http://rt.ets.org/hailsa/
 
- For understanding students' engagement with an assessment
- Method illustrations based on simulated data
- These features are used in the papers on the resouce page of the project (https://rt.ets.org/hailsa/resources.html)
                       
##### Please cite the appropriate studies and  references therein when you use or modify the codes 


## Table of Contents 
- [1 - Data Simulation](#section-1)
- [2 - Rapid-response Feature and Time Category](#section-2)
- [3 - Regularity-Index](#section-3)


In [ ]:
import math
import pandas as pd
import numpy as np
np.set_printoptions(precision=2, suppress=True)

import matplotlib
%matplotlib inline
%config InlineBackend.figure_format = 'svg'
import matplotlib.pyplot as plt
plt.style.use('ggplot')

import seaborn as sns
 

## <a id='section-1'></a>1 - Data Simulation




### Simulated Data 
**Note: parameter the selection is arbituary -- Suggest to estiamted parameters from real data**

For easy manipulations, we use von der Linden's hierarchy model (the accuracy and speed model, 2007; refer to the reference below).
- J = 15 items; multiple-choice problems scored either 0 or 1; each problem has five options.
- Session time limit is 30 minutes.
- N = 2000 test takers
#### Person Parameters
  - Population parameter: a person's ability $\theta\sim N(0,1)$; latent speed $\tau\sim N(0,1)$ (A higher $\tau$ results in a shorter response time).  Some studies on low-stakes large-scale assessments show that time effort (longer time) is seen to be  positively related to higher scores. Therefore, in this simulation, the Speed-Ability Correlation $\rho = - 0.2$. 
   
#### Item parameters  
- The core equation for time is $\ln(T_{ij}) \sim N(\beta_j - \tau_i, 1/\alpha_j^2)$.
- Accuracy Hyperparameters (assuming a 2PL model):Difficulty ($b$): $\mu_b = 0, \sigma_b = 1$ (Drawn from a Normal distribution).
- Discrimination ($a$): $\mu_{\ln(a)} = 0, \sigma_{\ln(a)} = 0.25$ (Drawn from a Lognormal distribution to ensure all slopes remain positive, typically ranging between 0.8 and 1.5).
   
- Time Hyperparameters:Time Intensity ($\beta$): $\mu_\beta = 4.1, \sigma_\beta = 0.4$. Rationale: $\beta$ is on the natural log scale.
      - An intercept of $4.8$ translates to $e^{4.8} \approx 121$ seconds. On a test of J=15, an average student may finish the test in 30 minutes.
- Time Discrimination ($\alpha$): $\mu_{\ln(\alpha)} = 0.5, \sigma_{\ln(\alpha)} = 0.2.$ 
   - Rationale: $\alpha$ represents the precision of the time distribution (variance is $1/\alpha^2$). The median value of $\alpha$ in this setup is $e^0.5 \sim 1.65$ that provides some "noise" around the expected response time.
- The Item Covariance ($\rho_{b, \beta}$): Difficulty-Intensity Correlation: 0.3 to 0.7.
     - Rationale: Harder items naturally take longer to read and solve. We draw $b$ and $\beta$ from a bivariate normal distribution with a strong positive correlation so the simulated hard items logically demand more time.

#### Simulation steps

The final simulation for a person $i$ taking item $j$ follows these precise equations:
- Step 1: Simulate the Score ($X_{ij}$)Calculate the probability of success using the 2PL equation, then draw the binary response from a Bernoulli distribution.$$P(X_{ij} = 1) = \frac{1}{1 + \exp(-a_j(\theta_i - b_j))}$$

  $$X_{ij} \sim \text{Bernoulli}(P(X_{ij} = 1))$$

- Step 2: Simulate the Time ($T_{ij}$)Calculate the expected log-time, add random noise based on the item's time precision, and then exponentiate it to get the time in real seconds.$$\ln(T_{ij}) = \beta_j - \tau_i + \varepsilon_{ij}$$(where $\varepsilon_{ij} \sim N(0, 1/\alpha_j^2)$)

 $$T_{ij} = \exp(\ln(T_{ij}))$$

- Total time is truncated to a 30-minute maximum.


## Reference:
van der Linden, W. J. (2007). A hierarchical framework for modeling speed and accuracy on test items. Psychometrika, 72(3), 287-308.
(The foundational paper defining the joint hierarchical model and its constant speed/conditional independence assumptions).

In [ ]:

# Set seed for reproducible synthetic data
np.random.seed(2026)

# ==========================================
# 1. Define Dimensions & Hyperparameters
# ==========================================
N = 2000  # Number of test takers
J = 15    # Number of items
MAX_TIME_SEC = 1800.0  # 30 minutes

# Person Hyperparameters
mu_person = [0, 0] # [mean_theta, mean_tau]
cov_person = [[1.0, -0.2],  # Var(theta), Cov(theta, tau)
              [-0.2, 1.0]]  # Cov(theta, tau), Var(tau)

# Item Hyperparameters
mu_b = 0.0
sd_b = 1.0
mu_beta = 3.5  # Adjusted so expected total time is ~30 mins; 4.8 is too high as 2 mins per item; 4.1 is ok as 1 min per item
sd_beta = 0.3 
corr_b_beta = 0.4
cov_b_beta = corr_b_beta * sd_b * sd_beta

mu_item = [mu_b, mu_beta]
cov_item = [[sd_b**2, cov_b_beta], 
            [cov_b_beta, sd_beta**2]]

# ==========================================
# 2. Simulate Latent Traits
# ==========================================
# Simulate Persons (Theta: Ability, Tau: Speed)
persons = np.random.multivariate_normal(mu_person, cov_person, N)
print(persons.shape)
print(f'First three: {persons[:3]}')
theta = persons[:, 0]
tau = persons[:, 1]

# Simulate Items (b: Difficulty, beta: Time Intensity)
items = np.random.multivariate_normal(mu_item, cov_item, J)
#print(f'Item b and beta parameters: {items}')
b = items[:, 0]
beta = items[:, 1]

# Simulate a (Discrimination) and alpha (Time Precision)
# Using lognormal so they are strictly positive
a = np.random.lognormal(mean=0.0, sigma=0.25, size=J)
alpha = np.random.lognormal(mean=0.8, sigma=0.15, size=J) ## mu=.5, sigma =.2 leads to too many nan; increased mean to .8
#print(f'a parameters: {a}')
#print(f' alpah parameters: {alpha}')


In [ ]:
# ==========================================
# 3. Simulate Full Responses and Times
# ==========================================
# Initialize matrices
X_full = np.zeros((N, J))
T_full = np.zeros((N, J))

for j in range(J):
    # IRT 2PL for Accuracy
    prob_correct = 1 / (1 + np.exp(-a[j] * (theta - b[j])))
    X_full[:, j] = np.random.binomial(1, prob_correct)
    
    # Lognormal for Time: ln(T) ~ N(beta - tau, 1/alpha^2)
    time_variance = 1 / (alpha[j]**2)
    log_time = np.random.normal(loc=(beta[j] - tau), scale=np.sqrt(time_variance))
    T_full[:, j] = np.exp(log_time)

# ==========================================
# 4. Apply the 30-Minute Truncation (Speededness)
# ==========================================
X_truncated = X_full.copy()
T_truncated = T_full.copy()

# Calculate cumulative time spent by each student
cumulative_time = np.cumsum(T_full, axis=1)

for i in range(N):
    # Find if and where the student exceeded 1800 seconds
    if cumulative_time[i, -1] > MAX_TIME_SEC:
        # Find the first item where they ran out of time
        cutoff_idx = np.argmax(cumulative_time[i] > MAX_TIME_SEC)
        
        # Calculate time remaining before starting this specific item
        time_spent_before = cumulative_time[i, cutoff_idx - 1] if cutoff_idx > 0 else 0
        time_left = MAX_TIME_SEC - time_spent_before
        
        # Truncate the time on the cutoff item to whatever was left
        T_truncated[i, cutoff_idx] = time_left
        
        # Standard psychometric assumption: If time ran out, the score is 0 (or NaN)
        # We will set it to NaN to indicate "Not Reached"
        X_truncated[i, cutoff_idx] = np.nan 
        
        # Nullify all subsequent items (they never saw them)
        if cutoff_idx + 1 < J:
            T_truncated[i, cutoff_idx + 1:] = np.nan
            X_truncated[i, cutoff_idx + 1:] = np.nan



In [ ]:
print(X_truncated[:3, :])
ISS = pd.DataFrame(X_truncated)
ISS = ISS.fillna(0)  ### remove nan
print(ISS.iloc[:3])

In [ ]:
#print(T_truncated[100:110, :])
ITT = pd.DataFrame(T_truncated)
ITT = ITT.fillna(0)
#print(ITT.iloc[100:110])

## <a id='section-2'></a>2 - Rapid-response Feature and Time Category


In [ ]:
from scipy.stats import spearmanr
import seaborn as sns
import matplotlib.pyplot as plt

print(f' High level visualization of the sumulated data:\n')

# Sample DataFrames with missing values
df01 = ITT ## ITT is the item response time matrix
df02 = ISS ## ISS is the response item score matrix (students are listed in the same order)

# Calculate row sums, ignoring missing values
x = df01.sum(axis=1, skipna=True) ##total response time
y = df02.sum(axis=1, skipna=True) ## total block score

# Calculate Spearman correlation
correlation, p_value = spearmanr(x, y)

print("Spearman Rank Correlation Coefficient between total score and total time:", round(correlation,2))
print("p-value:", round(p_value,2))


# Create scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(x, y, alpha=0.5)

# Fit a 3rd-degree polynomial
z = np.polyfit(x, y, 3)
p = np.poly1d(z)

# Create smooth x-values for the line
x_range = np.linspace(min(x), max(x), 100)

plt.plot(x_range, p(x_range), color="red", linewidth=2, label="Smooth Trend")
plt.xlabel("Row Sum of time")
plt.ylabel("Row Sum of score")
plt.title("Scatter Plot with Polynomial Fit")
plt.legend()
plt.grid(True)
plt.show()



#### The following function returns thresholds for flagging rapid responses based on three different methods 


- We suggest use the hybrid approach -- pick the smallest one to be conservative  

- please refer to the following papers for earlier development of these methods and their applications

 - Guo, H. & Ercikan. (2021, online first). Differential Rapid-Responding across Language and Cultural Groups.  Educational Research and Evaluation, https://www.tandfonline.com/eprint/XESAECFWMZWPKF5I6IDI/full?target=10.1080/13803611.2021.1963941
-  Rios,J.& Guo, H.(2020). Is there differential noneffortful responding between countries on an international assessment? Applied Measurement in Education, 33, 263-279.   DOI:10.1080/08957347.2020.1789141
-  Rios, J, Guo, H., etc. (2017). Evaluating the Impact of Careless Responding on Aggregated-Scores: To Filter Unmotivated Examinees or Not? International Journal of Testing, 17(1), 74–104. https://doi.org/10.1080/15305058.2016.1231193
 - Guo, H., Rios, J., Haberman, S., Liu, O. L., Wang, J.,& Paek, I. (2016). A new procedure for detection of students' rapid guessing responses using response time. Applied Measurement in Education, 29, 173-183.



In [ ]:
## define Chance score based on item context: we assuming that each item has 5 options for simplicity
## revise the chances based on your real items
chance = pd.Series(0.20, index=range(15))
#print(chance)
#print(chance.dtype)

In [ ]:
############# function for finding the threshold of a rapid responding ####
from sklearn.mixture import GaussianMixture
import pandas as pd
import numpy as np

##############################################################
# papers for the hybrid method for detecting rapid responses based on data
# 1. 2016 APM paper (CUMP)
# 2. 2020 APM paper (MLN) 
# 3. 2021 ERE paper (hybrid; NT method was cited in its references)

#####################################################
ULimit=100.0  ##upper limit of threshold in the cump approach; user's choice 
LLimit=5.0   ##lower limit of threshold in the cump approach; user's choice 

##### define CUMP
def CUMP_n(ITT, ISS, j, chance, ULimit, LLimit):
    """
    Calculates the response time threshold (y) based on cumulative probability.
    """
    # 1. Efficient Data Preparation
    # Select columns and convert to numeric in one go
    try:
        X = pd.concat([ITT.iloc[:, j], ISS.iloc[:, j]], axis=1)
        X.columns = ["time", "resp"]
        X = X.apply(pd.to_numeric, errors='coerce').dropna()
    except (IndexError, KeyError):
        return np.nan

    if X.empty:
        return np.nan

    # 2. Group by Time to find success counts per time point
    # We need counts of 0s and the total count (0s + 1s) per time point
    grouped = X.groupby("time")["resp"].agg(['count', 'sum'])
    # count is total (0+1), sum is the number of 1s. 
    # Therefore, (count - sum) is the number of 0s.
    zeros = grouped['count'] - grouped['sum']
    totals = grouped['count']
    
    # 3. Cumulative Probability Calculation
    # cum_con = 1 - (running sum of 0s / running sum of totals)
    cum_zeros = zeros.cumsum()
    cum_totals = totals.cumsum()
    cum_con = 1 - (cum_zeros / cum_totals)
    
    # 4. Filter by Chance and Find Threshold
    # We want the Time (S) where cumulative probability is <= chance[j]
    times = cum_con.index.values
    valid_mask = cum_con <= chance[j]
    
    if not valid_mask.any():
        return np.nan

    # Get the last time value that satisfies the condition
    value_T = times[valid_mask][-1]

    # 5. Apply Bounds
    if value_T > ULimit:
        return ULimit
    if value_T < LLimit:
        return np.nan
        
    return value_T

###################################################################
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from scipy.stats import norm

### define MLN
def MLN_n(ITT, ISS, j, chance, ULimit, LLimit):
    # 1. Clean and Log-transform Data
    X = pd.to_numeric(ITT.iloc[:, j], errors='coerce').dropna()
    X = X[X > 0] # Must be > 0 for log
    
    if len(X) < 10: # Avoid fitting GMM on insufficient data
        return np.nan
        
    Y = np.log(X).values.reshape(-1, 1)

    # 2. Fit GMM
    # n_init=3 helps find a better global optimum
    gmm = GaussianMixture(n_components=2, n_init=3, random_state=42).fit(Y)
    
    # 3. Extract and Sort Parameters (Ensure mu1 < mu2)
    means = gmm.means_.flatten()
    idx_sort = np.argsort(means)
    
    mu = means[idx_sort]
    sigmas = np.sqrt(gmm.covariances_.flatten()[idx_sort])
    weights = gmm.weights_[idx_sort]
    
    mu1, mu2 = mu[0], mu[1]
    sigma1, sigma2 = sigmas[0], sigmas[1]
    w1, w2 = weights[0], weights[1]

    # 4. Define Search Space in Log-Time
    # Search from the first mode to the second mode
    log_ULimit = np.log(ULimit)
    y_search = np.linspace(mu1, min(mu2, log_ULimit), 500)
    
    # 5. Compute Combined PDF
    pdf1 = w1 * norm.pdf(y_search, loc=mu1, scale=sigma1)
    pdf2 = w2 * norm.pdf(y_search, loc=mu2, scale=sigma2)
    combined_pdf = pdf1 + pdf2

    # 6. Find the Local Minimum (The Valley)
    # If the modes are too close, argmin will just pick an endpoint
    idx_min = np.argmin(combined_pdf)
    log_threshold = y_search[idx_min]
    
    # 7. Back-transform to original scale
    x_0 = np.exp(log_threshold)
    
    # 8. Apply Bounds
    if x_0 > ULimit:
        return ULimit
    if x_0 < LLimit:
        return np.nan
        
    return x_0


### Define the NT threshold
def NT(ITT, ISS, j, x):   ## x=10 or 20; experiment and choose x based on your real data
    X = ITT.iloc[:,j]
    X = pd.to_numeric(X, errors='coerce') ##The errors='coerce' argument will convert any non-numeric values to NaN (missing value) while converting the rest to a numeric type.
    X = X.dropna()
    y=np.percentile(X, x)
    return y



##################  The hybrid method: pick the smallest one
def hybrid_n(ITT, ISS, x, chance, ULimit, LLimit):  ## input ITT and ISS matrix, output time thresholds for all items
    J=ITT.shape[1]### test length
    cut=[0]*J  ## cuts
    for j in range(1, J+1):
        t_mln = MLN_n(ITT, ISS, j-1, chance, ULimit, LLimit)
        t_nt = NT(ITT, ISS, j-1,x)
        t_cump = CUMP_n(ITT, ISS, j-1, chance, ULimit, LLimit)
        y = np.nanmin([t_nt, t_mln, t_cump]) 
        y = y #np.asscalar(y) ## turn it into a scalar
        cut[j-1] = round(y,2)    
    return cut
   


In [ ]:
################################################################
 ###### Discretizing response time (six levels)
'''
 Recoding response time:

0: no time;
1: rapid response (RR),
2: RG - median;
3: median - 75;
4: 75 - 95;
5: above 95 (PL)
'''

### 1. cuts
def RTcut(ITT, ISS, x=20, me=50, q3=75, tl=95):
    RG = hybrid_n(ITT, ISS, x, chance, ULimit, LLimit)
    ITT = ITT.astype(float) ## from int64 to float64, so the quantiles can function !!!
    Median = ITT.median()
    Q3 = ITT.quantile(.75)
    TL = ITT.quantile(.95)   
    Tcut=np.column_stack([RG, np.round(Median, 2), np.round(Q3, 2), np.round(TL, 2)])  ## used column_stack if they are np array
    colnames = ["RR", "Median", "Q3", "95%tile"]
    Tcut = pd.DataFrame(Tcut, columns=colnames)
    return Tcut

print("Cuts for discretizing response times by item")
Tcut=RTcut(ITT, ISS, x=20, me=50, q3=75, tl=95)
print(Tcut)



#### Item response time: Categorized

In [ ]:
## 2. recode timing matrix df_Time to df_Time_new
import numpy as np

df_Time = ITT
n_row, n_col = df_Time.shape

# Extract Tcut values as a NumPy array
Tcut = np.array(Tcut) 
# Create RT.new as a NumPy array of zeros
df_Time_new0 = np.zeros((n_row, n_col))

# identify indices/locations for different cutoff values using NumPy boolean indexing; more powerful than loops
id_rp = np.where((df_Time.astype(float) <= Tcut[:, 0]) & (df_Time.astype(float) > 0))
id_50 = np.where((df_Time.astype(float) <= Tcut[:, 1]) & (df_Time.astype(float) > Tcut[:, 0]))
id_75 = np.where((df_Time.astype(float) <= Tcut[:, 2]) & (df_Time.astype(float) > Tcut[:, 1]))
id_95 = np.where((df_Time.astype(float) <= Tcut[:, 3]) & (df_Time.astype(float) > Tcut[:, 2]))
id_tl = np.where(df_Time.astype(float) > Tcut[:, 3])

# Assign values to RT.new using NumPy indexing
df_Time_new0[id_rp] = 1 ## RR
df_Time_new0[id_50] = 2 ## mid-low
df_Time_new0[id_75] = 3 ## mid-high
df_Time_new0[id_95] = 4 ## high
df_Time_new0[id_tl] = 5 ## prolonged

In [ ]:
df_Time_new = pd.DataFrame(df_Time_new0)
df_Time_new = df_Time_new.reset_index(drop=True)
print(df_Time_new.shape)
print(ITT.head(3))
print(df_Time_new.head(3))

## <a id='section-3'></a>3 - Regularity-Index


### Definition of Regularity Index
Based on the concept of entropy $H(x)$ - that quantifies the unpredictability or randomness of a sequence.
$$H(X) = - Σ [ p(x_i) * \log_b(p(x_i)) ]$$
where $p$ is the distribution of a random variable $X \in \{x_1, x_2, \cdots, x_n\}$.

We define the navigation regularity (or simply, Regularity) as:
$$
\mathrm{Regularity}(X) = \frac{1}{1+H(X)}.
$$

A higher value indicates a more regulated navigation pattern.

**Note:** In practice, we have navigation sequences. The $X$ above is the difference sequence of the navigations.
### Reference:
Guo, H., Johnson, M., Saldivia, L., Worthington, M., & Ercikan, K. (2025). Leveraging multi-AI agents to accelerate a teacher co-design. Proceeding of the 2025 Artificial Intelligence in Measurement and Education Conference (AIME-Con). https://aclanthology.org/2025.aimecon-main.4/

In [ ]:
import numpy as np
import math

def calculate_regularity(sequence):
    """
    Input: A sequence (list or np.array)
    Process: Calculates absolute differences (the pattern), then entropy.
    Output: (entropy, regularity)
    """
    # 1. Create the pattern: Absolute differences between consecutive values
    sequence = np.array(sequence)
    pattern = np.abs(np.diff(sequence))
    
    # Remove NaNs if any exist
    pattern = pattern[~np.isnan(pattern)]
    
    if len(pattern) == 0:
        return 0, 1.0

    # 2. Compute Frequencies
    unique, counts = np.unique(pattern, return_counts=True)
    probabilities = counts / len(pattern)
    
    # 3. Compute Entropy (using log2)
    entropy = -sum(p * math.log2(p) for p in probabilities)
    
    # 4. Compute Regularity
    regularity_score = 1 / (1 + entropy)
    
    return entropy, regularity_score



In [ ]:
### ------------------------------- examples ---------------------------
###Synthetic navigation sequences

import numpy as np

seq_1 = np.arange(1, 16)
# [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

# jump back and forth
seq_2 = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 7, 4, 5, 6, 7, 6, 7, 8, 9, 6, 7])

seq_3 = np.array([1, 2, 3, 1, 2, 1, 1, 2, 3, 2, 3, 4, 5, 6, 5, 6, 7, 8, 9, 7, 15])

#skips
seq_4 = np.array([1, 2, 5, 6, 8, 9, 10, 13, 14, 15])

#random
seq_5 = np.random.randint(1, 16, size=40)

ent, reg = calculate_regularity(seq_1)
print(f"seq_1 - Entropy: {ent:.2f}, Regularity: {reg:.2f}")

ent, reg = calculate_regularity(seq_2)
print(f"seq_2 - Entropy: {ent:.2f}, Regularity: {reg:.2f}")

ent, reg = calculate_regularity(seq_3)
print(f"seq_3 - Entropy: {ent:.2f}, Regularity: {reg:.2f}")

ent, reg = calculate_regularity(seq_4)
print(f"seq_4 - Entropy: {ent:.2f}, Regularity: {reg:.2f}")

ent, reg = calculate_regularity(seq_5)
print(f"seq_5 - Entropy: {ent:.2f}, Regularity: {reg:.2f}")

